In [1]:
import copy
import concurrent.futures
import pathlib
import dataclasses
import time

import numpy
import scipy.spatial

import cloudvolume
import kimimaro
import fastremap

np = numpy

In [2]:
import ac_pcg.label
import ac_pcg.chunks
import ac_pcg.skeletons

from ac_pcg.pcgraph.edges import Edges
from ac_pcg.pcgraph.edges import EDGE_TYPES

from ac_pcg.io.edges import put_chunk_edges
from ac_pcg.io.components import put_chunk_components

from ac_pcg.utils import (
    label_chunk,
    chunk_edges_from_skeleton
)

from ac_pcg.ac_processing.oversegment import (
    process_oversegment_array
)
from ac_pcg.ac_processing.utils.chunk_edges import (
    filter_edges_by_chunk,
    filter_edge_arr_by_vtx,
    chunk_edges_components_from_skeleton
)

In [3]:
# define inputs
data_path = pathlib.Path(
    "/ACdata0/Users/wanqing/data/projects/proofreading/output/H17_PO11_S8_20250408_mip0_pos3_large"
)

label_path = data_path / "segmentation_labeled.tif"
skeletons_path = data_path / "swcs/"

In [4]:
# load labeled array data
import imageio

labeled_array = imageio.v3.imread(label_path)
# labeled_array = labeled_array.astype('uint32')

In [5]:
%%time
def id_from_skel_path(skel_path):
    return int(skel_path.stem)


def load_skel_file(skel_path, skel_id=None):
    if skel_id is None:
        skel_id = id_from_skel_path(skel_path)     
    with skel_path.open(mode="r") as f:
        skel_str = f.read()
    skel = cloudvolume.skeleton.Skeleton.from_swc(skel_str)
    skel.id = skel_id
    return skel


# load skeleton data
def load_skeleton_directory(skels_path):
    skel_paths_iter = skels_path.iterdir()
    
    skels_result = {
        skel.id: skel for skel in (
            load_skel_file(skel_path)
            for skel_path in skel_paths_iter
        )
    }

    # FIXME skel not hashable, so concurrent loading doesn't work.
    # with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as e:
    #     futs = e.map(load_skel_file, skel_paths_iter)
    #     skels_result = {}
    #     for fut in concurrent.futures.as_completed(futs):
    #         skel = fut.result()
    #         skels_result[skel.id] = skel

    return skels_result

skels = load_skeleton_directory(skeletons_path)

CPU times: user 809 ms, sys: 298 ms, total: 1.11 s
Wall time: 2.49 s


In [6]:
# optionally, convert/transform skeletons to match dimensionality of label array
#   here, will do xyz -> zyx

def vtx_shift_cvskel(cvskel, convert_int=False):
    new_cvskel = copy.deepcopy(cvskel)
    new_cvskel.vertices = new_cvskel.vertices[:, ::-1]
    if convert_int:
        new_cvskel.vertices = numpy.around(new_cvskel.vertices).astype(int)
    return new_cvskel

def relabel_skeletons(skels, relabel_func=None):
    new_skels = copy.deepcopy(skels)
    if relabel_func is None:
        return new_skels
    relabeled_skels = {}
    for sk_id, new_skel in new_skels.items():
        new_sk_id = relabel_func(sk_id)
        new_skel.id = new_sk_id
        relabeled_skels[new_sk_id] = new_skel
    return relabeled_skels

# swcs are output 0-indexed relative to the nonzero label values
#   and with xyz vs zyx axis order.
#   Correct those to generate "label_skels"
label_skels = {
    sk_id: vtx_shift_cvskel(sk)
    for sk_id, sk in relabel_skeletons(
        skels, relabel_func=lambda x: int(x)+1
    ).items()
}

In [7]:
%%time
# TODO generate chunk skeleton id mappings as a method

labeled_array_chunksize = numpy.array((128, 128, 128))

skel_id_to_bboxes = {sk_id: cloudvolume.Bbox.from_points(sk.vertices) for sk_id, sk in label_skels.items()}

skel_to_chunks = {sk_id: ac_pcg.chunks.get_bbox_chunks(sk_bbox, labeled_array_chunksize) for sk_id, sk_bbox in skel_id_to_bboxes.items()}

chunk_to_skel_ids = {}
for sk_id, sk_chunks in skel_to_chunks.items():
    for sk_chunk in sk_chunks:
        try:
            chunk_to_skel_ids[tuple(sk_chunk)].append(sk_id)
        except KeyError:
            chunk_to_skel_ids[tuple(sk_chunk)] = [sk_id]

CPU times: user 394 ms, sys: 10.6 ms, total: 405 ms
Wall time: 385 ms


In [8]:
# run oversegmentation over chunks with skeletons
labeler = ac_pcg.label.ChunkLabeler()

chunk_size = labeled_array_chunksize
# chunk_boxes = chunk_to_skel_ids.keys()
chunk_boxes = (
    ac_pcg.chunks.chunk_idx_to_bbox(chunk_idx, chunk_size, labeled_array.shape)
    for chunk_idx in chunk_to_skel_ids.keys()
)
output_arr = numpy.empty(labeled_array.shape, dtype=numpy.uint64)

output_skels = copy.deepcopy(label_skels)

oversegment_kwargs = {
    "downsample": 3,
    "progress": False,
}

tic = time.time()
for chunk_num, chunk_box in enumerate(chunk_boxes):
    chunk_contains_bb = cloudvolume.Bbox(
        chunk_box.bbox.minpt,
        chunk_box.bbox.maxpt - 1
    )
    subvol_arr = labeled_array[chunk_box.bbox.to_slices()]
    skels_indices_tuples = filter(None, (
        ac_pcg.skeletons.bboxed_skel(label_skels[skel_id], chunk_contains_bb)
        for skel_id in chunk_to_skel_ids.get(chunk_box.chunk_idx, [])
    ))
    try:
        subvol_skels, subvol_skel_indices = zip(*skels_indices_tuples)
    except ValueError:
        continue
    oversegmented_subvol_arr, oversegmented_subvol_skels = process_oversegment_array(
        subvol_arr, subvol_skels, lambda x: labeler.encode_chunk_seg(chunk_box.chunk_idx, x),
        oversegment_kwargs=oversegment_kwargs, relabel_zero=False
    )
    
    output_arr[chunk_box.bbox.to_slices()] = oversegmented_subvol_arr[...]

    # map new indices to original skel vertices
    for i, (skel, subvol_indices) in enumerate(zip(oversegmented_subvol_skels, subvol_skel_indices)):
        output_skel = output_skels[skel.id]
        try:
            output_skel.segments[subvol_indices] = skel.segments
        except AttributeError:
            output_skel.add_vertex_attribute(
                "segments",
                numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
            )
            output_skel.segments[subvol_indices] = skel.segments

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

print(time.time() - tic)

0 0.18983983993530273
10 1.643718957901001
20 3.102536201477051
30 4.492585897445679
40 5.898855447769165
50 7.277452707290649
60 8.565860748291016
70 9.906503200531006
80 11.130879878997803
90 12.57042121887207
100 13.836896657943726
110 15.170852184295654
120 16.45935821533203
130 17.715088605880737
140 19.104291200637817
150 20.285333156585693
160 21.578462600708008
170 23.007111310958862
180 24.3244206905365
190 25.548062086105347
25.67913246154785


In [9]:
# there is an edge case here which does not generate segment values.
#   Ignore these skeletons and figure that out later
output_skels = {sk_id: skel for sk_id, skel in output_skels.items() if hasattr(skel, "segments")}

print(list(label_skels.keys() - output_skels.keys()))

[3482]


In [10]:
%%time

skel_id_to_unique_vtxs_idxs_locs = {}
skel_id_to_full_array_idxs = {}

offset = 0

for skel_id, skel in output_skels.items():
    skel_id_to_unique_vtxs_idxs_locs[skel_id] = ac_pcg.skeletons.skeleton_unique_vtxs_idxs_locs(skel)
    vtx_size = skel_id_to_unique_vtxs_idxs_locs[skel_id].vertices.size
    skel_id_to_full_array_idxs[skel_id] = numpy.arange(offset, offset + vtx_size)
    offset += vtx_size
    

CPU times: user 253 ms, sys: 8.27 ms, total: 262 ms
Wall time: 249 ms


In [11]:
%%time

all_vtxs, all_idxs, all_locs = zip(*((v.vertices, v.indices, v.locations) for k, v in skel_id_to_unique_vtxs_idxs_locs.items()))

all_vtxs = numpy.concatenate(all_vtxs)
all_idxs = numpy.concatenate(all_idxs)
all_locs = numpy.concatenate(all_locs)

all_kdtree = scipy.spatial.KDTree(all_locs)

CPU times: user 16.7 ms, sys: 34 μs, total: 16.7 ms
Wall time: 15.4 ms


In [12]:
%%time
# generate edges between skeletons enforcing non-directional uniqueness

distance = 30
edge_set = set()

for i, (skel_id, vtxs_idxs_locs) in enumerate(skel_id_to_unique_vtxs_idxs_locs.items()):
    skel_tree = scipy.spatial.KDTree(vtxs_idxs_locs.locations)
    pairs = skel_tree.query_ball_tree(all_kdtree, r=distance)
    in_skel_ids = set(skel_id_to_full_array_idxs[skel_id])

    for skel_idx, pair_result in enumerate(pairs):
        skel_vtx = vtxs_idxs_locs.vertices[skel_idx]
        edge_set |= {
            frozenset((skel_vtx, all_vtxs[query_vtx_idx]))
            for query_vtx_idx in (set(pair_result) - in_skel_ids)
            if (skel_vtx != all_vtxs[query_vtx_idx])
        }
    if not i % 1000:
        print(i)

0
1000
2000
3000
4000
CPU times: user 675 ms, sys: 19.9 ms, total: 695 ms
Wall time: 692 ms


In [13]:
%time all_pairs = numpy.array([tuple(edge) for edge in edge_set])
all_pairs.shape

CPU times: user 94.9 ms, sys: 130 μs, total: 95 ms
Wall time: 93.7 ms


(120737, 2)

In [14]:
# define outputs
output_root_path = pathlib.Path("/allen/programs/celltypes/workgroups/em-connectomics/russelt/processing_in_progress/axonal_connectomics/ac_pcg_output/260211_trial/")
edge_output_path = output_root_path / "edges"
component_output_path = output_root_path / "components"
label_output_path = output_root_path / "supervoxel_labels"

label_output_cv_info = {
    "data_type": "uint64",
    "num_channels": 1,
    "scales": [
        {
            "chunk_sizes": [
                [
                    128,
                    128,
                    128
                ]
            ],
            "compressed_segmentation_block_size": (8, 8, 8),
            "encoding": "compressed_segmentation",
            "key": "650_748_748",
            "resolution": [
                650,
                748,
                748
            ],
            "size": output_arr.shape,
            "voxel_offset": [
                0,
                0,
                0
            ]
        }
    ],
    "type": "segmentation"
}

In [15]:
%%time

# edge_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/processing_in_progress/axonal_connectomics/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/edges"
# component_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/processing_in_progress/axonal_connectomics/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/components"

edge_output_loc = str(edge_output_path)
component_output_loc = str(component_output_path)

 # NOTE -- there are zeros in components and edges -- filter from full edge list here
all_pairs = filter_edge_arr_by_vtx(all_pairs, 0)

tic = time.time()

for chunk_num, (chunk, chunk_skel_ids) in enumerate(chunk_to_skel_ids.items()):
    in_chunk_edge_results = []
    between_chunk_edge_results = []
    connected_components_results = []
    # get connected components and active edges for all skeletons
    for skel_id in chunk_skel_ids:
        try:
            skel = output_skels[skel_id]
        except KeyError:
            # if not in output_skels, skip
            continue
        chunk_edge_components = chunk_edges_components_from_skeleton(skel, chunk, labeler)

        if chunk_edge_components.in_chunk_edges is not None:
            in_chunk_edge_results.append(chunk_edge_components.in_chunk_edges)
        if chunk_edge_components.between_chunk_edges is not None:
            between_chunk_edge_results.append(chunk_edge_components.between_chunk_edges)
        if chunk_edge_components.chunk_components is not None:
            connected_components_results.append(chunk_edge_components.chunk_components)

   

    # get inactive edges from spatial query results
    chunk_inactive_edges = filter_edges_by_chunk(all_pairs, chunk, labeler)

    in_chunk_edge_results.append(chunk_inactive_edges.in_chunk_edges)
    between_chunk_edge_results.append(chunk_inactive_edges.between_chunk_edges)
    
    in_chunk_edge_arr = numpy.concatenate(in_chunk_edge_results)
    between_chunk_edge_arr = numpy.concatenate(between_chunk_edge_results)

    in_chunk_edges = Edges(*in_chunk_edge_arr.T)
    between_chunk_edges = Edges(*between_chunk_edge_arr.T)
    # cross_chunk_edges are not generated in this segmentation method
    cross_chunk_edges = Edges([], [])

    # produce chunk edge format
    edges_d = {
        EDGE_TYPES.in_chunk: in_chunk_edges,
        EDGE_TYPES.between_chunk: between_chunk_edges,
        EDGE_TYPES.cross_chunk: cross_chunk_edges
    }

    # serialization handles writing connected components format
    connected_components = connected_components_results

    has_edges = len(in_chunk_edge_results) or len(between_chunk_edge_results)
    has_components = len(connected_components)

    if has_edges:
        put_chunk_edges(edge_output_loc, chunk, edges_d, compression_level=22)
    if has_components:
        put_chunk_components(component_output_loc, connected_components, chunk)

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

0 0.033347129821777344
10 0.3605825901031494
20 0.7120182514190674
30 1.0576527118682861
40 1.4000053405761719
50 1.968672513961792
60 2.387943983078003
70 2.770373821258545
80 3.110424280166626
90 3.514590263366699
100 3.9059410095214844
110 4.278168678283691
120 4.649693250656128
130 5.020364761352539
140 5.408609628677368
150 5.778036594390869
160 6.178501129150391
170 6.622234344482422
180 7.064386606216431
190 7.441559791564941
CPU times: user 5.27 s, sys: 160 ms, total: 5.43 s
Wall time: 7.49 s


In [16]:
label_output_loc = str(label_output_path)
seg_cv = cloudvolume.CloudVolume(label_output_loc, info=label_output_cv_info)


In [17]:
seg_cv.commit_info()
seg_cv[..., 0] = output_arr[...]

Uploading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 192/192 [00:15<00:00, 12.65it/s]
